In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# For plotting to verify
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

gen_details = pd.read_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv')
hw_tseries = pd.read_csv('/scratch/ng72/ms5578/time_series/gen_hw_status.csv')
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
# Select start and end dates (Jul 2009 - Jun 2024)
sdate, edate = '2009-07-01','2024-06-30'

In [4]:
# Select frequency of timeseries (hourly or daily)
mode = 'hourly'

In [5]:
# Select region or fuel type (optional)

def select_group(gen_details, state=None, ftype=None):
    if state is not None and ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[(gen_details['region'] == state) & (gen_details['fuel_source_primary'].isin(ftype))]
        else:
            groups = gen_details.groupby(['region', 'fuel_source_primary'])
            grp = groups.get_group((state, ftype))
    elif state is not None:
        groups = gen_details.groupby('region')
        grp = groups.get_group(state)
    elif ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[gen_details['fuel_source_primary'].isin(ftype)]
        else:
            groups = gen_details.groupby('fuel_source_primary')
            grp = groups.get_group(ftype)
    else:
        grp = gen_details

    return grp

info = select_group(gen_details,ftype=['Wind','Solar']).copy()

The functions below retrieve the timeseries for the specified dates.
The data is resampled to daily if specified above. The df is merged with the generation information.

In [29]:
def process_group(grp, gen_fpath, hw_tseries, start_date=None, end_date=None, mode='daily'):

    # Sanitize DUIDs and build file paths
    safe_duids = [duid.replace("/", "_").replace("\\", "_") for duid in grp['DUID']]
    gen_locs = [f"{gen_fpath}/{duid}.csv" for duid in safe_duids]
    dfs = [pd.read_csv(fp, dtype='object') for fp in gen_locs if os.path.exists(fp)]
    print(f"Loaded {len(dfs)} CSV file(s) out of {len(gen_locs)} expected.")

    if not dfs:
        raise ValueError("no files to load.")

    # Concatenate and clean header rows
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # Type conversions
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)

    # Filter by date
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index('time').sort_index()
    hw_tseries = hw_tseries.loc[start_date:end_date]

    if mode == 'daily':
        # Aggregate dfs to daily
        agg_func = {'TOTALMWh': 'sum', 'TOTALCLEARED': 'sum', 'AGCSTATUS': 'max'}
        dfs_daily = dfs.groupby(['DUID', pd.Grouper(freq='1D')]).agg(agg_func).reset_index()
        hw_tseries_daily = hw_tseries.reset_index()
        # Normalize time columns to midnight for exact matching
        dfs_daily['time'] = pd.to_datetime(dfs_daily['time']).dt.normalize()
        hw_tseries_daily['time'] = pd.to_datetime(hw_tseries_daily['time']).dt.normalize()
        # Merge on DUID and time
        merged = pd.merge(
            dfs_daily,
            hw_tseries_daily,
            on=['DUID', 'time'],
            how='left'
        )
    elif mode == 'hourly':
        dfs_hourly = dfs.reset_index()
        # Create a full hourly time index for each DUID
        duids = dfs_hourly['DUID'].unique()
        time_range = pd.date_range(start=start_date, end=end_date, freq='1h')
        full_index = pd.MultiIndex.from_product([duids, time_range], names=['DUID', 'time'])
        hw_tseries_daily = hw_tseries.reset_index().set_index(['DUID', 'time'])
        # Reindex to hourly and forward-fill
        hw_tseries_hourly = hw_tseries_daily.reindex(full_index).groupby(level=0).ffill().groupby(level=0).bfill().reset_index()
        # Merge on DUID and time
        merged = pd.merge(
            dfs_hourly,
            hw_tseries_hourly,
            on=['DUID', 'time'],
            how='left'
        )
    else:
        raise ValueError("mode must be 'daily' or 'hourly'")

    merged = merged.dropna(how='all')

    # Merge with info DataFrame
    df = merged.merge(
        grp[['DUID', 'fuel_source_primary', 'region']],
        on='DUID',
        how='left'
    )

    return df.reset_index(drop=True)

In [ ]:
# Make sure mode is set correctly to daily or hourly
df = process_group(info, gen_fpath, hw_tseries, sdate, edate, mode=mode)
df

Loaded 164 CSV file(s) out of 216 expected.


In [30]:
daily = process_group(info, gen_fpath, hw_tseries, sdate, edate, mode='daily')
daily

Loaded 164 CSV file(s) out of 216 expected.


,DUID,time,TOTALMWh,TOTALCLEARED,AGCSTATUS,lat,lon,tas,EHF_val,HW_EHF_avg,HW_EHF_peak,EHF_flag,tas_3d_avg,tas_3d_peak,event_group,HW_event_day,technology_type_primary,fuel_source_primary,region
0,ADPPV1,2021-04-20,0.000000,0.000000,0,-35.09,138.53,285.169189,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Solar PV - Fixed,Solar,SA1
1,ADPPV1,2021-04-21,0.000000,0.000000,0,-35.09,138.53,286.577393,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Solar PV - Fixed,Solar,SA1
2,ADPPV1,2021-04-22,0.000000,0.000000,0,-35.09,138.53,288.400391,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Solar PV - Fixed,Solar,SA1
3,ADPPV1,2021-04-23,0.000000,0.000000,0,-35.09,138.53,287.435547,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Solar PV - Fixed,Solar,SA1
4,ADPPV1,2021-04-24,0.000000,0.000000,0,-35.09,138.53,286.481689,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Solar PV - Fixed,Solar,SA1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302810,YENDWF1,2024-06-26,2459.171760,2480.919936,0,-37.62,144.03,279.865234,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind Turbine - Onshore,Wind,VIC1
302811,YENDWF1,2024-06-27,1019.514172,1032.626931,0,-37.62,144.03,279.619141,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind Turbine - Onshore,Wind,VIC1
302812,YENDWF1,2024-06-28,2719.304275,2730.902507,0,-37.62,144.03,282.438232,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind Turbine - Onshore,Wind,VIC1
302813,YENDWF1,2024-06-29,1574.405059,1589.164169,0,-37.62,144.03,278.438232,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind Turbine - Onshore,Wind,VIC1


The following section contains all the data cleaning functions and their execution. These should be selected based on technology type and research question.